# PNG/BMP SRNet — Colab ortam hazırlığı

Çalışma zamanı türü: **Python 3 / CPU / Runtime Version 2026.07**.
GPU seçmeden bu tek kontrol hücresini çalıştırın.
Bu notebook kurulu paket sürümlerini ve okunabilen RAM/swap sınırlarını gösterir.
Paket kurmaz, Torch import etmez, GPU bağlamı veya model oluşturmaz.

Daha önce tamamlanan 23 Ekim kapasite notebook'unu yeniden çalıştırmayın.
6 Kasım WSL ölçümü kendi ortamına aittir; Colab süre tahmini sayılmaz.

[Google runtime sürümleri](https://research.google.com/colaboratory/runtime-version-faq.html)


In [ ]:
import importlib.metadata
import json
import platform
from pathlib import Path, PurePosixPath

result = {"schema": "colab-cpu-readiness-v1", "python": platform.python_version(),
          "gpu_calls": 0, "models_created": 0, "packages": {}}
for package in ("numpy", "Pillow", "torch", "onnx", "onnxruntime"):
    try:
        result["packages"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        result["packages"][package] = "unavailable"

try:
    with Path("/proc/self/cgroup").open("rb") as stream:
        raw = stream.read(4097)
    if len(raw) > 4096:
        raise ValueError("cgroup bound")
    identities = [line[3:] for line in raw.decode().splitlines() if line.startswith("0::")]
    if len(identities) != 1:
        raise ValueError("cgroup identity")
    relative = PurePosixPath(identities[0])
    if not relative.is_absolute() or ".." in relative.parts or len(relative.parts) > 32:
        raise ValueError("cgroup identity")
    folder = Path("/sys/fs/cgroup").joinpath(*relative.parts[1:])
    counters = {}
    for key, name in (("ram_limit_bytes", "memory.max"),
                      ("swap_limit_bytes", "memory.swap.max")):
        path = folder / name
        if any(p.is_symlink() for p in (path, *path.parents)):
            raise ValueError("cgroup ancestry")
        with path.open("rb") as stream:
            value = stream.read(33)
        if len(value) > 32:
            raise ValueError("counter bound")
        text = value.decode().strip()
        if text == "max":
            counters[key] = "unbounded"
        elif text.isascii() and text.isdecimal():
            counters[key] = int(text)
        else:
            raise ValueError("counter type")
    result["cgroup"] = {"status": "observed_metadata_only", **counters}
except (OSError, UnicodeError, ValueError):
    result["cgroup"] = {"status": "unavailable"}

expected = {"python": "3.12.13", "numpy": "2.0.2", "Pillow": "11.3.0", "torch": "2.11.0+cpu"}
observed = {"python": result["python"], **result["packages"]}
result["target_runtime"] = "2026.07"
result["package_metadata_matches"] = all(observed.get(k) == v for k, v in expected.items())
result["status"] = ("metadata_ready_only" if result["package_metadata_matches"]
                    else "unavailable_pinned_environment")
result["profile_attempt_started"] = False
print(json.dumps(result, indent=2, sort_keys=True))


## Sonuç ve devam

CPU runtime için beklenen Torch paketi `2.11.0+cpu` olur; CUDA paketi ayrı GPU ortamında doğrulanacaktır.
`metadata_ready_only` yalnız Python/paket metadatasının beklenen sürümlerle eşleştiğini söyler.
GPU uyumluluğunu, kaynak denetimini, süre/bellek sınırlarını veya eğitim hazırlığını kanıtlamaz.
Uyumsuz ortam `unavailable_pinned_environment` kalır; otomatik paket kurulumu yapılmaz.

Sonraki hazırlık Colab'a uygun bağımsız süreç denetimi, kaynak/veri hash paketi ve
kesintide tüketim kayıtlarını koruyan ayrı bir ölçüm sözleşmesidir.
Bu notebook gerçek veri yüklemez ve model eğitmez. Çalışmayı bitirince CPU oturumunu kapatın.
